In [ ]:
import numpy as np
import pandas as pd


In [ ]:
anime_data=pd.read_csv('../Data/anime.csv')

In [ ]:
anime_data.head()

In [ ]:
anime_data.tail()

In [ ]:
anime_data.shape

In [ ]:
anime_data.info()

In [ ]:
anime_data.dtypes

In [ ]:
anime_data.describe()

In [ ]:
anime_data.columns

In [ ]:
print(anime_data[['Score', 'Ranked',
                  'Score-1', 'Score-2', 'Score-3', 'Score-4',
                  'Score-5', 'Score-6', 'Score-7', 'Score-8',
                  'Score-9', 'Score-10']].dtypes)

In [ ]:
anime_data.isnull().sum()

In [ ]:
anime_data.duplicated().sum()

In [ ]:
filtering_unknown=(anime_data=="Unknown").sum()
filtering_unknown

#### Checking the data value for each column to determine the type of data type for it. I used `transpose` because normally it would display the columns but not all would be visible due to its length

In [ ]:
anime_data.head().T

| Data-quality check          | Finding |
| --------------------------- | ------- |
| Number of rows              | 17562       |
| Number of columns           | 35    |
| Duplicate rows              | 0       |
| Columns with missing values | 0       |
| Number of Columns containig `Unknown`| 25      |


### Data Quality Assessment

The dataset contains 17,562 rows and 35 columns. No duplicate rows were identified, and Pandas reported no explicit `NaN` values.

However, further investigation revealed that 25 columns contain `"Unknown"` values. The number of unknown entries varies substantially across columns, with `Premiered`, `Licensors`, and `English name` containing particularly large numbers of unknown values.

The presence of `"Unknown"` does not necessarily indicate invalid data. Each affected column will therefore be evaluated individually to determine whether the value should be retained as a category, converted to `NaN`, or handled through another appropriate cleaning strategy.

#### Before any convertion, we want to ensure that the columns to be converted has no other error apart from the Unknown

In [ ]:
anime_data['Score'].unique()

In [ ]:
anime_data['Score'].unique()[:20]

In [ ]:
anime_data['Score'].unique()[-20:]

In [ ]:
anime_data["Score"].value_counts().head(20)

In [ ]:
anime_data["Ranked"].unique()

In [ ]:
anime_data['Ranked'].value_counts()

In [ ]:
anime_data["Score-10"].head()

In [ ]:
anime_data['Score-10'].unique()[20:50]

In [ ]:
anime_data['Score-10'].value_counts()

In [ ]:
anime_data['Score-10'].dtypes

#### Testing whether unknown is the only missing value before conversion of Score-10

In [ ]:
score_columns=["Score-1","Score-2","Score-3","Score-4","Score-5","Score-6","Score-7","Score-8","Score-9","Score-10"]
for columns in score_columns:
    conversion=pd.to_numeric(anime_data[columns], errors='coerce').isna().sum()
    error=(anime_data[columns] == 'Unknown').sum()

    print(columns,"",conversion)
    print(error,"\n")

#### From our observation above, Score-1 to Score-10 has not other errors in them apart from `Unknown`






#### Test whether `Unknown` is the only thing preventing the columns, Ranked and Score from being numerical

In [ ]:
for column in ['Score',"Ranked"]:
    unknown=(anime_data[column]=="Unknown").sum()
    non_numerical=pd.to_numeric(anime_data[column], errors='coerce').isna().sum()

    print(column)
    print("Unknown: ",unknown)
    print("Non-numeric: ", non_numerical)

##### Our observation above depicts that the only error existing in the columns `Score` and `Ranked` is `Unknown`. 

## DATA CLEANING

In [ ]:
anime_data_clean=anime_data.copy()
anime_data_clean['Score']=anime_data_clean['Score'].replace('Unknown', np.nan)
anime_data_clean['Ranked']=anime_data_clean['Ranked'].replace('Unknown', np.nan)


score_cols=["Score-1","Score-2","Score-3","Score-4","Score-5","Score-6","Score-7","Score-8","Score-9","Score-10"]
for cols in score_cols:
    anime_data_clean[cols]=anime_data_clean[cols].replace('Unknown', np.nan)



In [ ]:
anime_data_clean['Score']=anime_data_clean['Score'].astype(float)
anime_data_clean['Ranked']=pd.to_numeric(anime_data_clean['Ranked'], errors="coerce").astype("Int64")

for col in score_cols:
    anime_data_clean[col]=pd.to_numeric(anime_data_clean[col]).astype("Int64")


In [ ]:
anime_data_clean.info()

In [ ]:
anime_data_clean.columns=anime_data_clean.columns.str.strip().str.replace(" ","_")

In [ ]:
unknown_cols = [
    'Genres',
    'English_name',
    'Japanese_name',
    'Type',
    'Episodes',
    'Aired',
    'Premiered',
    'Producers',
    'Licensors',
    'Studios',
    'Source',
    'Duration',
    'Rating'
]

anime_data_clean[unknown_cols].dtypes

In [ ]:
(anime_data_clean[unknown_cols]=="Unknown").sum().sort_values(ascending=False)

In [ ]:
anime_data_clean["Episodes"]=anime_data_clean['Episodes'].replace("Unknown", np.nan)
anime_data_clean['Episodes']=anime_data_clean["Episodes"].astype("Int64")

#### Observing columns `Aired`, `Premiered`,`Duration`,`Rating` to decide the approriate datatype to assign

In [ ]:
anime_data_clean[['Aired','Premiered','Duration','Rating']].head(10)

#### Checking for format inconsistencies

In [ ]:
for col in ['Aired','Premiered','Duration','Rating']:
    print(f"\n------{col}-----")
    print(anime_data_clean[col].value_counts().head(10))

#### From our observation from above, we would maintain the columns `Aired`, `Premiered`,`Rating` but look into `Duration`

In [ ]:
anime_data_clean['Duration'].value_counts()

In [ ]:
anime_data[anime_data_clean['Duration'].str.contains("hr", case=False, na=True)]['Duration'].value_counts().sort_values(ascending=False)

In [ ]:
anime_data_clean['Duration'].value_counts().tail(20)

##### We have successfully discovered that the `Duration` column had its duration in hours, minuites and seconds. Our next code would extract all the numbers and convert them into minuites for uniformity